In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import files
files.upload()

# structure du dataset

In [ ]:
df = pd.read_csv('/content/hotel_bookings.csv')
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

# Valeurs manquantes (en nombre et en %)

In [ ]:
manquants = df.isna().sum()
pct = (manquants / len(df) * 100).round(2)
pd.DataFrame({'nb_manquants': manquants, 'pct': pct})[manquants > 0].sort_values('pct', ascending=False)

**Valeurs manquantes :** `company` est vide à 94 % → l'absence signifie « pas d'entreprise , on remplace par 0.
`agent` (14 %) → absence = « pas d'agent », remplacé par 0. `country` (488 lignes, 0,4 %) → remplacé par « Unknown ».


# Valeurs aberrantes

In [ ]:
print("adr négatif :", (df['adr'] < 0).sum())
print("adr > 1000 :", (df['adr'] > 1000).sum())
print("0 nuit :", ((df['stays_in_weekend_nights'] + df['stays_in_week_nights']) == 0).sum())
print("0 personne :", ((df['adults'] + df['children'].fillna(0) + df['babies']) == 0).sum())
print(df['meal'].value_counts())

**Valeurs aberrantes :**
- 2 prix incohérents (un négatif, un au-dessus de 1 000 €) : supprimés.
- 180 réservations sans aucun client : supprimées.
- 715 séjours de 0 nuit : gardés, ce sont peut-être des locations à la journée.
- Repas « Undefined » : regroupé avec « SC » (sans repas), qui veut dire la même chose.

# Cible et répartition des hôtels

In [ ]:
print(df['is_canceled'].value_counts(normalize=True).round(3))
print(df['hotel'].value_counts())

In [ ]:
df.duplicated().sum()

**Doublons :** 31 994 lignes identiques, soit 27 % du total. Le fichier n'a pas de numéro de réservation, donc ces lignes peuvent être de vraies réservations différentes (par exemple un groupe qui réserve plusieurs chambres pareilles). On les garde, sinon on fausse le taux d'annulation.

# nettoyage

In [ ]:
df_clean = df.copy()

# Valeurs manquantes
df_clean['children'] = df_clean['children'].fillna(0).astype(int)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0).astype(int)
df_clean['company'] = df_clean['company'].fillna(0).astype(int)

# Incohérences
df_clean = df_clean[(df_clean['adults'] + df_clean['children'] + df_clean['babies']) > 0]
df_clean = df_clean[(df_clean['adr'] >= 0) & (df_clean['adr'] < 1000)]
df_clean['meal'] = df_clean['meal'].replace('Undefined', 'SC')

# Format date
df_clean['reservation_status_date'] = pd.to_datetime(df_clean['reservation_status_date'])

print(df.shape, "→", df_clean.shape)
print("Manquants restants :", df_clean.isna().sum().sum())

**Bilan du nettoyage :** on passe de 119 390 à 119 208 réservations (182 lignes supprimées, soit 0,15 %). Il ne reste plus aucune valeur manquante. Les colonnes connues seulement après la réservation (`reservation_status`, `assigned_room_type`…) sont gardées pour l'analyse et le SQL ; elles seront retirées seulement pour le modèle.

# describe() pour repérer d’autres valeurs extrêmes  important

In [ ]:
pd.crosstab(df_clean['reservation_status'], df_clean['is_canceled'])


# Pourquoi 1 000 € comme seuil ?

In [ ]:
print(df['adr'].sort_values(ascending=False).head(8).values)
print(df['adr'].quantile([0.5, 0.9, 0.99, 0.999]).round(1))

In [ ]:
top = df['adr'].sort_values(ascending=False).head(15).reset_index(drop=True)
couleurs = ['#e34948' if v > 1000 else '#2a78d6' for v in top]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(range(1, 16), top, color=couleurs, edgecolor='white', linewidth=2)
ax.axhline(1000, color='#555555', linestyle='--', linewidth=1.5, label='Seuil : 1 000 €')
for i, v in enumerate(top, start=1):
    ax.text(i, v + 60, f'{v:.0f} €', ha='center', fontsize=8)
ax.set_title('Les 15 prix par nuit les plus élevés : une seule valeur aberrante',
             loc='left', fontweight='bold')
ax.set_xlabel('Rang (du prix le plus élevé au 15ᵉ)')
ax.set_ylabel('Prix par nuit (€)')
ax.set_xticks(range(1, 16))
ax.legend(frameon=False); sns.despine()
plt.tight_layout(); plt.savefig('fig_seuil_prix.png', dpi=150); plt.show()

**Pourquoi un seuil de 1 000 € ?** Le graphique montre qu'une seule valeur (5 400 €) se détache complètement. Le prix suivant n'est que de 510 €, soit 10 fois moins, et 99 % des prix sont inférieurs à 252 €. Le seuil de 1 000 € isole donc uniquement cette erreur, sans toucher aux prix élevés mais réalistes.
La méthode IQR (règle des « moustaches ») aurait classé environ 3 800 réservations comme aberrantes. Ce sont surtout de vraies grandes chambres en haute saison : nous ne l'avons donc pas appliquée.

#  Prix par type de chambre et par hôtel

In [ ]:
prix_chambre = (df_clean.groupby(['hotel', 'reserved_room_type'])['adr']
                .agg(nb='size', prix_median='median', prix_max='max').round(0))
prix_chambre

In [ ]:
COULEURS = {'City Hotel': '#2a78d6', 'Resort Hotel': '#eb6834'}

In [ ]:
prix = (df_clean.groupby(['hotel', 'reserved_room_type'])['adr']
        .agg(nb='size', mediane='median').reset_index())
prix = prix[prix['nb'] >= 50]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (hotel, d) in zip(axes, prix.groupby('hotel')):
    ax.bar(d['reserved_room_type'], d['mediane'], color=COULEURS[hotel], edgecolor='white', linewidth=2)
    for x, y in zip(d['reserved_room_type'], d['mediane']):
        ax.text(x, y + 3, f'{y:.0f} €', ha='center', fontsize=9)
    ax.set_title(hotel, loc='left', fontweight='bold')
    ax.set_xlabel('Type de chambre')
axes[0].set_ylabel('Prix médian par nuit (€)')
fig.suptitle('Prix médian par nuit selon le type de chambre', x=0.01, ha='left', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.savefig('fig_prix_chambres.png', dpi=150); plt.show()

**Pourquoi un seuil de 1 000 € ?** Le graphique montre qu'une seule valeur (5 400 €) se détache complètement. Le prix suivant n'est que de 510 €, soit 10 fois moins, et les suivants descendent progressivement jusqu'à 384 €. Le seuil de 1 000 € isole donc uniquement cette erreur, sans toucher aux prix élevés mais réalistes.
La méthode IQR (règle des « moustaches ») aurait classé environ 3 800 réservations comme aberrantes. Ce sont surtout de vraies grandes chambres en haute saison : nous ne l'avons donc pas appliquée.

**Prix par type de chambre :** au City Hotel, le prix médian augmente régulièrement de A (94 €) à G (222 €). Au Resort Hotel, la tendance est la même de A (61 €) à H (184 €), sauf la chambre C (163 €), plus chère que D, E et F. Les codes de chambre sont anonymisés : ils ne correspondent pas forcément à une gamme, ni à la même chambre d'un hôtel à l'autre.
La chambre A est la plus réservée et la moins chère dans les deux hôtels (médiane de 94 € et 61 €). Les 5 400 € supprimés représentaient donc environ 57 fois le prix normal : c'est bien une erreur. Les prix élevés, jusqu'à 510 €, restent plausibles et sont gardés.
Les types de chambre trop rares (moins de 50 réservations : C au City, B et L au Resort) ne sont pas affichés.

#  Les prix à 0 € : pourquoi on les garde

In [ ]:
print("Réservations à 0 € :", (df_clean['adr'] == 0).sum())
df_clean.loc[df_clean['adr'] == 0, 'market_segment'].value_counts()

**Prix à 0 € :** 1 810 réservations (1,5 % du total). 665 sont des séjours offerts par l'hôtel (« Complementary »), ce qui explique le prix nul. Les autres sont réparties dans tous les segments : il peut s'agir de promotions, de compensations ou de prix non renseignés. Comme elles sont peu nombreuses et pas forcément fausses, elles sont gardées.

In [ ]:
print(df['is_canceled'].value_counts(normalize=True).round(3))
print(df['hotel'].value_counts())

**Cible :** 37 % des réservations sont annulées et 63 % ne le sont pas. Les classes sont déséquilibrées, mais pas trop.
**Hôtels :** le City Hotel a 2 fois plus de réservations (79 330) que le Resort Hotel (40 060).

In [ ]:
df.duplicated().sum()

**Doublons :** 31 994 lignes identiques, soit 27 % du total. Le fichier n'a pas de numéro de réservation, donc ces lignes peuvent être de vraies réservations différentes (par exemple un groupe qui réserve plusieurs chambres pareilles). On les garde, sinon on fausse le taux d'annulation.

# Créer des colonnes utiles pour la suite

In [ ]:
mois = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,'July':7,
        'August':8,'September':9,'October':10,'November':11,'December':12}
df_clean['arrival_month_num'] = df_clean['arrival_date_month'].map(mois)

df_clean['arrival_date'] = pd.to_datetime(dict(year=df_clean['arrival_date_year'],
                                               month=df_clean['arrival_month_num'],
                                               day=df_clean['arrival_date_day_of_month']))
df_clean['total_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']

df_clean[['arrival_date', 'arrival_month_num', 'total_nights', 'total_guests']].head()

# Vérifier que les dates sont bien de vraies dates

In [ ]:
df_clean[['arrival_date', 'reservation_status_date']].dtypes

# Remettre les autres colonnes au bon type

In [ ]:
# Mois dans l'ordre (janvier → décembre) pour les graphiques
ordre_mois = list(mois.keys())
df_clean['arrival_date_month'] = pd.Categorical(df_clean['arrival_date_month'],
                                                categories=ordre_mois, ordered=True)

# Colonnes oui/non (0/1) : on vérifie qu'elles ne contiennent que 0 et 1
for col in ['is_canceled', 'is_repeated_guest']:
    print(col, sorted(df_clean[col].unique()))

df_clean.dtypes

children, agent, company → int64 (et non plus float64)
arrival_date, reservation_status_date → datetime64[ns]
arrival_date_month → category

# afficher les dates en français

In [ ]:
df_clean['arrival_date'].dt.strftime('%d/%m/%Y').head()

**Formats :** les dates (`arrival_date`, `reservation_status_date`) sont converties au format date standard AAAA-MM-JJ, compatible avec BigQuery. Les mois sont rangés dans l'ordre du calendrier, et `children`, `agent` et `company` sont passés en nombres entiers.

In [ ]:
pd.crosstab(df_clean['reservation_status'], df_clean['is_canceled'])

In [ ]:
df_clean.describe().T

In [ ]:
(df_clean['adults'] == 0).sum()

**Réservations sans adulte :** 223 réservations contiennent seulement des enfants (0,2 %). C'est possible, par exemple une chambre séparée réservée pour les enfants d'une famille : on les garde.

In [ ]:
pd.crosstab(df_clean['reservation_status'], df_clean['is_canceled'])

## Étude des annulations.

# Par établissement

In [ ]:
par_hotel = df_clean.groupby('hotel').agg(
    reservations=('is_canceled', 'size'),
    annulations=('is_canceled', 'sum'),
    taux_annulation=('is_canceled', 'mean')
)
par_hotel['taux_annulation'] = (par_hotel['taux_annulation'] * 100).round(1)
par_hotel

**Par établissement :** le City Hotel annule beaucoup plus que le Resort Hotel : **41,8 %** contre **27,8 %**, soit 14 points d'écart.
Le City Hotel concentre aussi 3 annulations sur 4 (33 078 sur 44 198). C'est l'établissement où l'enjeu est le plus fort.

###  Volume et taux d'annulation par hôtel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vol = df_clean['hotel'].value_counts()
tx = df_clean.groupby('hotel')['is_canceled'].mean().mul(100).round(1).reindex(vol.index)
axes[0].bar(vol.index, vol.values, color=[COULEURS[h] for h in vol.index])
axes[0].set_title('Nombre de réservations', loc='left', fontweight='bold')
axes[1].bar(tx.index, tx.values, color=[COULEURS[h] for h in tx.index])
axes[1].axhline(MOY, color='#555555', linestyle='--', label=f'Moyenne ({MOY:.0f} %)')
axes[1].set_title("Taux d'annulation (%)", loc='left', fontweight='bold')
axes[1].legend(frameon=False)
for i, v in enumerate(vol):
    axes[0].text(i, v, f'{v:,}'.replace(',', ' '), ha='center', va='bottom')
for i, v in enumerate(tx):
    axes[1].text(i, v, f'{v} %', ha='center', va='bottom')
sns.despine(); plt.tight_layout(); plt.savefig('fig1_hotels.png', dpi=150); plt.show()

**Constat :** le City Hotel reçoit **2 fois plus de réservations** que le Resort Hotel (79 162 contre 40 046), mais ce n'est pas la seule raison de ses nombreuses annulations : son **taux** est aussi bien plus élevé (**41,8 %** contre **27,8 %**, au-dessus de la moyenne de 37 %). Le City Hotel annule donc plus **en nombre ET en proportion** : c'est l'établissement le plus exposé.

# Graphique 2 : saisonnalité par mois d’arrivée

### 2. Taux d'annulation selon le mois d'arrivée

In [ ]:
mensuel = df_clean.groupby(['arrival_date_month', 'hotel'], observed=True)['is_canceled'].mean().mul(100).unstack()
ax = mensuel.plot(marker='o', linewidth=2, color=[COULEURS[c] for c in mensuel.columns], figsize=(10, 4.5))
ax.axhline(MOY, color='#555555', linestyle='--', linewidth=1.5, label=f'Moyenne ({MOY:.0f} %)')
ax.set_title("Taux d'annulation par mois d'arrivée", loc='left', fontweight='bold')
ax.set_ylabel("Taux d'annulation (%)"); ax.set_xlabel('')
ax.set_xticks(range(12)); ax.set_xticklabels([m[:3] for m in mensuel.index])
ax.legend(frameon=False); sns.despine()
plt.tight_layout(); plt.savefig('fig2_mois.png', dpi=150); plt.show()
mensuel.round(1)

**Constat : deux profils saisonniers différents.**
- **City Hotel :** le taux reste **au-dessus de la moyenne toute l'année** (entre 37 % et 46 %). Il est le plus élevé au **printemps** (avril : 46 %, mai-juin : environ 44 %). Le risque est donc permanent, quelle que soit la saison.
- **Resort Hotel :** le taux suit la **saison touristique**. Il est faible en hiver (janvier : 15 %, novembre : 19 %) et monte en **été** (juin-septembre : 31 % à 33 %), quand la demande est forte et que les clients réservent longtemps à l'avance.
- Le Resort Hotel reste **sous la moyenne tous les mois**.

→ Au Resort Hotel, la vigilance doit porter sur les **mois d'été** ; au City Hotel, elle doit être **constante**, avec un pic au printemps.

# Graphique 3 : évolution de juillet 2015 à août 2017

### 3. Évolution du taux d'annulation dans le temps

In [ ]:
evol = df_clean.groupby([df_clean['arrival_date'].dt.to_period('M'), 'hotel'])['is_canceled'].mean().mul(100).unstack()
evol.index = evol.index.to_timestamp()
ax = evol.plot(linewidth=2, color=[COULEURS[c] for c in evol.columns], figsize=(11, 4.5))
ax.set_title("Évolution mensuelle du taux d'annulation", loc='left', fontweight='bold')
ax.set_ylabel("Taux d'annulation (%)"); ax.set_xlabel('')
ax.legend(frameon=False); sns.despine()
plt.tight_layout(); plt.savefig('fig3_evolution.png', dpi=150); plt.show()

**Constat :**
- **City Hotel :** le taux est très élevé au début des données (67 % en juillet 2015), puis se stabilise **entre 35 % et 50 %**, avec des pics au printemps 2016 et 2017. Le point de départ très élevé est à interpréter avec prudence : il correspond au début de la collecte des données.
- **Resort Hotel :** le taux suit un rythme saisonnier régulier, avec des creux chaque hiver (environ 13 % en janvier 2016, 16 % en janvier 2017). Surtout, il **augmente nettement en 2017**, jusqu'à environ 38 %, au point de **dépasser le City Hotel en juillet-août 2017**.

→ L'écart entre les deux hôtels se réduit en 2017 : la hausse des annulations au Resort Hotel est un **signal d'alerte** à surveiller.

### 4. Taux d'annulation selon le délai de réservation

In [ ]:
df_clean['lead_time_tranche'] = pd.cut(df_clean['lead_time'], bins=[-1, 7, 30, 90, 180, 365, 800],
                                       labels=['0-7 j', '8-30 j', '1-3 mois', '3-6 mois', '6-12 mois', '+1 an'])
taux_par_hotel('lead_time_tranche', "Taux d'annulation selon le délai de réservation", 'fig4_delai.png')

**Constat : plus on réserve longtemps à l'avance, plus on annule.** C'est vrai dans les deux hôtels, et la progression est régulière.
- Réservé **moins d'une semaine avant** : très peu d'annulations (12 % au City Hotel, environ 6 % au Resort Hotel).
- Réservé **plus d'un an avant** : **71 %** d'annulations au City Hotel et environ 46 % au Resort Hotel.
- Au City Hotel, le taux dépasse la moyenne dès **1 à 3 mois** d'avance ; au Resort Hotel, à partir de **3 à 6 mois**.

→ Le délai de réservation est **l'un des facteurs les plus liés à l'annulation**. C'est logique : plus l'arrivée est lointaine, plus le client a le temps de changer de plan ou de trouver une meilleure offre. Cette variable, connue au moment de la réservation, sera **très utile pour le modèle**.

### 5. Taux d'annulation selon le type d'acompte

In [ ]:
taux_par_hotel('deposit_type', "Taux d'annulation par type d'acompte", 'fig5_acompte.png')

**Constat : un résultat surprenant.**
- **Sans acompte** (« No Deposit », 88 % des réservations) : 30,5 % d'annulations au City Hotel, 24,7 % au Resort Hotel.
- **Acompte non remboursable** (« Non Refund ») : **99,8 %** d'annulations au City Hotel et **96 %** au Resort Hotel.
- **Acompte remboursable** : seulement 142 réservations au Resort Hotel (15,5 %), et trop peu au City Hotel pour être affiché.

**Ce paradoxe s'explique probablement** par le fait que ces réservations non remboursables sont surtout des **réservations de groupe ou d'agences**, qui bloquent des chambres à l'avance puis les libèrent. Les données ne permettent pas de le vérifier précisément. Ce résultat est donc à interpréter avec **prudence** : il ne faut pas en conclure qu'un acompte non remboursable « provoque » des annulations.

### 6. Taux d'annulation selon le segment de marché

In [ ]:
taux_par_hotel('market_segment', "Taux d'annulation par segment de marché", 'fig6_segment.png',
               horizontal=True, min_n=100)

**Constat :**
- **Groupes** : le segment le plus risqué dans les deux hôtels, surtout au City Hotel (**68,9 %** contre 42,4 %).
- **Agences en ligne** (Online TA) : le plus gros volume (56 408 réservations), avec un taux proche de la moyenne et **identique dans les deux hôtels** (environ 36 %).
- **Agences hors ligne** (Offline TA/TO) : **le plus grand écart entre les deux hôtels**, 42,9 % au City Hotel contre seulement 15,2 % au Resort Hotel.
- **Clients directs, entreprises et séjours offerts** : moins de 22 % d'annulations, ce sont les clients les plus fiables.
- Le segment **Aviation** (équipages) n'existe qu'au City Hotel.

→ Au City Hotel, les **groupes** et les **agences hors ligne** sont les priorités. Au Resort Hotel, ce sont les **groupes** et les **agences en ligne**.

### 7. Taux d'annulation selon le type de client

In [ ]:
taux_par_hotel('customer_type', "Taux d'annulation par type de client", 'fig7_type_client.png')

**Constat :**
- **Transient** (client individuel, la grande majorité des réservations) : environ 46 % d'annulations au City Hotel, contre 31 % au Resort Hotel.
- **Contract** (contrat avec une entreprise ou une agence) : **le plus grand écart**, environ 48 % au City Hotel contre seulement 9 % au Resort Hotel.
- **Group** : environ 10 % dans les deux hôtels.
- **Transient-Party** (individuels liés à un groupe) : environ 28 % au City Hotel et 20 % au Resort Hotel.

→ Le City Hotel annule plus pour presque tous les types de clients. Les clients sous contrat sont fiables au Resort Hotel, mais pas au City Hotel.

### 8. Taux d'annulation selon l'historique du client

In [ ]:
df_clean['historique'] = np.select(
    [df_clean['previous_cancellations'] > 0, df_clean['is_repeated_guest'] == 1],
    ['A déjà annulé', 'Client fidèle'], default='Nouveau client')
taux_par_hotel('historique', "Taux d'annulation selon l'historique du client", 'fig8_historique.png',
               ordre=['Client fidèle', 'Nouveau client', 'A déjà annulé'])

**Constat : le passé du client est le signal le plus fort.**
- **A déjà annulé** : environ **93 %** d'annulations au City Hotel et 84 % au Resort Hotel.
- **Nouveau client** : proche de la moyenne (environ 38 % au City Hotel, 27 % au Resort Hotel).
- **Client fidèle** : presque aucune annulation (environ 5 % au City Hotel, 1 % au Resort Hotel).

→ Un client qui a déjà annulé recommencera très probablement. Un client fidèle vient presque toujours. Cette information est connue au moment de la réservation, donc **utilisable par le modèle**.

### 9. Taux d'annulation selon le nombre de demandes spéciales

In [ ]:
df_clean['nb_demandes'] = df_clean['total_of_special_requests'].clip(upper=3).astype(str).replace('3', '3+')
taux_par_hotel('nb_demandes', "Taux d'annulation selon le nombre de demandes spéciales", 'fig9_demandes.png',
               ordre=['0', '1', '2', '3+'])

**Constat :**
- **Aucune demande** : 55 % d'annulations au City Hotel, 32,3 % au Resort Hotel.
- **Dès 1 demande** : le taux tombe à environ **22 % dans les deux hôtels**.
- **3 demandes ou plus** : environ 17 %.

→ Un client qui fait une demande (lit bébé, étage, vue…) est **impliqué dans son séjour** et annule moins. Au City Hotel, une réservation sans demande annule 2,5 fois plus.
⚠️ Ces demandes peuvent être ajoutées **après** la réservation : à discuter avant de les utiliser dans le modèle.

# 2. Par segment de marché

In [ ]:
par_segment = df_clean.groupby('market_segment').agg(
    reservations=('is_canceled', 'size'),
    taux_annulation=('is_canceled', 'mean')
)
par_segment['taux_annulation'] = (par_segment['taux_annulation'] * 100).round(1)
par_segment.sort_values('taux_annulation', ascending=False)

**Par segment de marché :**
- Les **groupes** annulent le plus : **61,1 %**, presque 2 fois la moyenne.
- Les **agences de voyage** (en ligne 36,8 %, hors ligne 34,3 %) sont proches de la moyenne, mais représentent le plus gros volume (plus de 80 000 réservations).
- Les clients **directs** (15,4 %), **entreprises** (18,8 %) et **séjours offerts** (12,2 %) annulent peu : ce sont les clients les plus fiables.
- Le segment « Undefined » ne compte que 2 réservations : on ne l'interprète pas.

In [ ]:
seg = par_segment[par_segment['reservations'] >= 100].sort_values('taux_annulation')
moy = df_clean['is_canceled'].mean() * 100
couleurs = ['#e34948' if t > moy else '#2a78d6' for t in seg['taux_annulation']]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(seg.index, seg['taux_annulation'], color=couleurs)
ax.axvline(moy, color='#555555', linestyle='--', linewidth=1)
ax.text(moy, len(seg) - 0.45, f' moyenne {moy:.1f} %', color='#555555', fontsize=9)
for i, (t, n) in enumerate(zip(seg['taux_annulation'], seg['reservations'])):
    ax.text(t + 0.8, i, f"{t} %  ({n:,} résa)".replace(',', ' '), va='center', fontsize=9)

ax.set_xlabel("Taux d'annulation (%)")
ax.set_xlim(0, 75)
ax.set_title("Taux d'annulation par segment de marché", loc='left', fontweight='bold')
sns.despine()
plt.tight_layout()
plt.savefig('fig_segment_global.png', dpi=150, bbox_inches='tight')
plt.show()

→ Le graphique 6 détaille ces segments **hôtel par hôtel** : l'écart le plus fort concerne les agences hors ligne (42,9 % au City Hotel contre 15,2 % au Resort Hotel).

# 3. Segment × hôtel (pour comparer les deux établissements)

In [ ]:
seg_hotel = pd.crosstab(df_clean['market_segment'], df_clean['hotel'],
                        values=df_clean['is_canceled'], aggfunc='mean').mul(100).round(1)
seg_hotel

### Vérification des prix aberrants
On affiche les 2 réservations dont le prix par nuit (`adr`) est négatif ou supérieur à 1 000 €, pour vérifier qu'il s'agit bien d'erreurs avant de les supprimer.



In [ ]:
cols = ['hotel', 'is_canceled', 'reserved_room_type', 'adults', 'stays_in_weekend_nights',
        'stays_in_week_nights', 'adr', 'arrival_date_month', 'arrival_date_year', 'market_segment']
df.loc[(df['adr'] < 0) | (df['adr'] > 1000), cols]

**Constat :** les 2 lignes sont bien des erreurs de saisie.
- **5 400 €** pour **1 nuit** en chambre A (la moins chère) au City Hotel, alors que le prix médian de cette chambre est de 94 €. La réservation a d'ailleurs été annulée.
- **-6,38 €** pour un séjour de 10 nuits d'un groupe au Resort Hotel : un prix négatif est impossible.

→ Ces 2 lignes sont supprimées.

# Cellule de préparation graphiques

In [ ]:
sns.set_theme(style='whitegrid')
COULEURS = {'City Hotel': '#2a78d6', 'Resort Hotel': '#eb6834'}
MOY = df_clean['is_canceled'].mean() * 100

def taux_par_hotel(col, titre, fichier, ordre=None, horizontal=False, min_n=50):
    """Taux d'annulation par catégorie, comparé entre les deux hôtels."""
    t = df_clean.groupby([col, 'hotel'], observed=True)['is_canceled'].agg(['mean', 'size']).reset_index()
    t = t[t['size'] >= min_n]                      # on écarte les catégories trop petites
    taux = t.pivot(index=col, columns='hotel', values='mean').mul(100).round(1)
    nb = t.pivot(index=col, columns='hotel', values='size')
    if ordre is not None:
        taux = taux.reindex([o for o in ordre if o in taux.index])
    elif horizontal:
        taux = taux.loc[taux.mean(axis=1).sort_values(ascending=False).index]
    nb = nb.reindex(taux.index)

    ax = taux.plot(kind='barh' if horizontal else 'bar', color=[COULEURS[c] for c in taux.columns],
                   figsize=(10, 4.5), width=0.75, edgecolor='white', linewidth=2)
    if horizontal:
        ax.axvline(MOY, color='#555555', linestyle='--', linewidth=1.5, label=f'Moyenne ({MOY:.0f} %)')
        ax.set_xlabel("Taux d'annulation (%)"); ax.set_ylabel(''); ax.invert_yaxis()
    else:
        ax.axhline(MOY, color='#555555', linestyle='--', linewidth=1.5, label=f'Moyenne ({MOY:.0f} %)')
        ax.set_ylabel("Taux d'annulation (%)"); ax.set_xlabel(''); plt.xticks(rotation=0)
    ax.set_title(titre, loc='left', fontweight='bold')
    ax.legend(frameon=False)
    sns.despine()
    plt.tight_layout(); plt.savefig(fichier, dpi=150, bbox_inches='tight'); plt.show()
    return taux.join(nb, lsuffix=' (%)', rsuffix=' (nb)')

# Qualité des données : graphique des valeurs manquantes

In [ ]:
manq = (df.isna().mean() * 100).round(2)
manq = manq[manq > 0].sort_values()
ax = manq.plot(kind='barh', color='#2a78d6', figsize=(8, 3))
for i, v in enumerate(manq):
    ax.text(v + 1, i, f'{v} %', va='center')
ax.set_title('Part de valeurs manquantes par colonne (données brutes)', loc='left', fontweight='bold')
ax.set_xlabel('% de valeurs manquantes')
sns.despine(); plt.tight_layout(); plt.savefig('fig0_manquants.png', dpi=150); plt.show()

# Les deux hôtels : volume ET taux (deux graphiques côte à côte)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vol = df_clean['hotel'].value_counts()
tx = df_clean.groupby('hotel')['is_canceled'].mean().mul(100).round(1).reindex(vol.index)
axes[0].bar(vol.index, vol.values, color=[COULEURS[h] for h in vol.index])
axes[0].set_title('Nombre de réservations', loc='left', fontweight='bold')
axes[1].bar(tx.index, tx.values, color=[COULEURS[h] for h in tx.index])
axes[1].axhline(MOY, color='#555555', linestyle='--', label=f'Moyenne ({MOY:.0f} %)')
axes[1].set_title("Taux d'annulation (%)", loc='left', fontweight='bold'); axes[1].legend(frameon=False)
for ax, serie in zip(axes, [vol, tx]):
    for i, v in enumerate(serie):
        ax.text(i, v, f'{v:,.0f}'.replace(',', ' ') if serie is vol else f'{v} %', ha='center', va='bottom')
sns.despine(); plt.tight_layout(); plt.savefig('fig1_hotels.png', dpi=150); plt.show()

# Saisonnalité : taux par mois d’arrivée

In [ ]:
mensuel = df_clean.groupby(['arrival_date_month', 'hotel'], observed=True)['is_canceled'].mean().mul(100).unstack()
ax = mensuel.plot(marker='o', linewidth=2, color=[COULEURS[c] for c in mensuel.columns], figsize=(10, 4.5))
ax.axhline(MOY, color='#555555', linestyle='--', linewidth=1.5, label=f'Moyenne ({MOY:.0f} %)')
ax.set_title("Taux d'annulation par mois d'arrivée", loc='left', fontweight='bold')
ax.set_ylabel("Taux d'annulation (%)"); ax.set_xlabel('')
ax.set_xticks(range(12)); ax.set_xticklabels([m[:3] for m in mensuel.index])
ax.legend(frameon=False); sns.despine()
plt.tight_layout(); plt.savefig('fig2_mois.png', dpi=150); plt.show()
mensuel.round(1)

#  Prix de la nuit

In [ ]:
plt.figure(figsize=(10, 4.5))
ax = sns.boxplot(data=df_clean, x='hotel', y='adr', hue='is_canceled', showfliers=False,
                 palette={0: '#1baf7a', 1: '#e34948'})
ax.set_title("Prix moyen par nuit selon l'annulation", loc='left', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Prix par nuit (€)')
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ['Venu', 'Annulé'], frameon=False)
sns.despine(); plt.tight_layout(); plt.savefig('fig11_prix.png', dpi=150); plt.show()
df_clean.groupby(['hotel', 'is_canceled'])['adr'].median()

# Pays : top 10 par nombre de réservations

In [ ]:
top10 = df_clean['country'].value_counts().head(10).index
pays = (df_clean[df_clean['country'].isin(top10)].groupby('country')['is_canceled']
        .agg(taux='mean', nb='size').assign(taux=lambda d: (d['taux'] * 100).round(1))
        .sort_values('taux'))
ax = pays['taux'].plot(kind='barh', color='#2a78d6', figsize=(9, 4.5))
ax.axvline(MOY, color='#555555', linestyle='--', label=f'Moyenne ({MOY:.0f} %)')
ax.set_title("Taux d'annulation des 10 pays qui réservent le plus", loc='left', fontweight='bold')
ax.set_xlabel("Taux d'annulation (%)"); ax.set_ylabel(''); ax.legend(frameon=False)
sns.despine(); plt.tight_layout(); plt.savefig('fig12_pays.png', dpi=150); plt.show()
pays.sort_values('nb', ascending=False)

## Analyse approfondie des annulations
Pour chaque question, on compare le **taux d'annulation** des deux hôtels (City Hotel en bleu, Resort Hotel en orange). La ligne en pointillés indique la moyenne générale (37 %).

# Parking

In [ ]:
df_clean['parking'] = np.where(df_clean['required_car_parking_spaces'] > 0, 'Avec parking', 'Sans parking')
taux_par_hotel('parking', "Taux d'annulation selon la demande de parking", 'fig10_parking.png',
               ordre=['Sans parking', 'Avec parking'])

# Prix par nuit : réservations venues vs annulées

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=df_clean, x='hotel', y='adr', hue='is_canceled',
            palette={0: '#1baf7a', 1: '#e34948'}, showfliers=False, ax=ax)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ['Venu', 'Annulé'], frameon=False)
ax.set_title("Prix par nuit selon l'annulation", loc='left', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Prix par nuit (€)')
sns.despine(); plt.tight_layout()
plt.savefig('fig11_prix_annulation.png', dpi=150, bbox_inches='tight'); plt.show()
df_clean.groupby(['hotel', 'is_canceled'])['adr'].median().unstack().round(1)

# Corrélations avec l’annulation

In [ ]:
corr = (df_clean.select_dtypes('number').drop(columns=['agent', 'company'])
        .corr()['is_canceled'].drop('is_canceled').sort_values())
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(corr.index, corr.values, color=['#e34948' if v > 0 else '#1baf7a' for v in corr.values])
ax.axvline(0, color='#555555', linewidth=1)
ax.set_title("Corrélation de chaque variable avec l'annulation", loc='left', fontweight='bold')
sns.despine(); plt.tight_layout()
plt.savefig('fig13_correlations.png', dpi=150, bbox_inches='tight'); plt.show()
corr.round(2)

# Synthèse : quelles variables sont liées à l’annulation ?

In [ ]:
num = df_clean.select_dtypes('number').drop(columns=['is_canceled', 'agent', 'company'])
corr = num.corrwith(df_clean['is_canceled']).sort_values()
ax = corr.plot(kind='barh', color=np.where(corr > 0, '#e34948', '#1baf7a'), figsize=(9, 7))
ax.axvline(0, color='#555555', linewidth=1)
ax.set_title("Corrélation de chaque variable avec l'annulation", loc='left', fontweight='bold')
ax.set_xlabel('Corrélation (rouge = augmente le risque, vert = le diminue)')
sns.despine(); plt.tight_layout(); plt.savefig('fig13_correlations.png', dpi=150); plt.show()
corr.round(3)

**Comparaison des deux hôtels par segment :**
- Le City Hotel annule plus dans **presque tous les segments**.
- Le plus gros écart concerne les **agences hors ligne** : **42,9 %** au City Hotel contre seulement **15,2 %** au Resort Hotel.
- Les **groupes** annulent beaucoup dans les deux hôtels, mais surtout au City Hotel (**68,9 %** contre 42,4 %).
- Les **agences en ligne** se comportent de la même façon dans les deux hôtels (environ 36 %).
- Le segment « Aviation » (équipages de compagnies aériennes) n'existe qu'au City Hotel, ce qui est logique pour un hôtel en ville.

**À retenir pour la direction :** les réservations de **groupe** et d'**agences hors ligne au City Hotel** sont les plus exposées à l'annulation. Ce sont des cibles prioritaires pour demander un acompte ou relancer le client.

**Fuite de données :** `reservation_status` correspond exactement à la cible. « Canceled » (42 992) et « No-Show » (1 206) donnent toujours `is_canceled = 1`, « Check-Out » (75 010) donne toujours 0.
Cette variable n'est connue qu'**après** l'arrivée prévue : si on la donnait au modèle, il « tricherait » avec 100 % de réussite. Elle sera donc **exclue du modèle**, tout comme `reservation_status_date`.
On remarque aussi que 1 206 annulations sont des **no-shows** : le client n'est pas venu, sans prévenir.

## Synthèse : principaux constats

1. **Le City Hotel annule beaucoup plus** : 41,8 % contre 27,8 % au Resort Hotel. Il concentre 3 annulations sur 4.
2. **La saison joue surtout au Resort Hotel** : 15 % en janvier, plus de 31 % en été. Le City Hotel reste au-dessus de la moyenne toute l'année (37 à 46 %).
3. **Plus on réserve tôt, plus on annule** : de 12 % (moins d'une semaine avant) à 71 % (plus d'un an avant) au City Hotel.
4. **Les acomptes non remboursables annulent presque tous** (99,8 % au City Hotel, 96 % au Resort Hotel), probablement à cause des réservations de groupes ou d'agences.
5. **Les groupes sont le segment le plus risqué** (68,9 % au City Hotel). Les agences hors ligne montrent le plus grand écart entre les deux hôtels (42,9 % contre 15,2 %).
6. **L'historique du client est le signal le plus fort** : environ 90 % d'annulations chez ceux qui ont déjà annulé, presque aucune chez les clients fidèles.
7. **Un client qui fait au moins une demande spéciale annule moins** : environ 22 % contre 55 % sans demande au City Hotel.

→ Pour le modèle, les variables les plus prometteuses sont le délai de réservation, le type d'acompte, le segment, l'historique du client et l'hôtel.

## Conclusion générale

### 1. Les données
Le jeu de données réunit **119 390 réservations** de deux hôtels portugais, un hôtel urbain (City Hotel) et un hôtel de vacances (Resort Hotel), pour des arrivées entre juillet 2015 et août 2017. **37 % des réservations sont annulées.**

### 2. Ce que nous avons vérifié
Avant de nettoyer, nous avons contrôlé la qualité des données sous quatre angles :
- **la structure** : dimensions, types de colonnes, signification des variables ;
- **les valeurs manquantes** : 4 colonnes concernées ;
- **les doublons** : 31 994 lignes identiques ;
- **les incohérences et valeurs extrêmes** : prix négatifs, réservations sans client, séjours de 0 nuit, valeurs très élevées.

### 3. Le nettoyage réalisé, et pourquoi

**Valeurs manquantes : remplacées, pas supprimées**
- `company` (94 % vide) et `agent` (14 % vide) : une case vide ne signifie pas une erreur, mais que la réservation s'est faite **sans entreprise** ou **sans agence**. Nous avons remplacé par 0 pour garder cette information.
- `country` (488 lignes) : remplacé par « Unknown ». Supprimer ces lignes aurait fait perdre des réservations pour une seule information manquante.
- `children` (4 lignes) : remplacé par 0, la valeur de loin la plus fréquente.

**Lignes incohérentes : supprimées (182 lignes, soit 0,15 %)**
- 180 réservations **sans aucun client** (0 adulte, 0 enfant, 0 bébé) : une réservation sans personne n'a pas de sens.
- 2 **prix aberrants** (un négatif, un supérieur à 1 000 €) : erreurs de saisie évidentes, qui auraient faussé les moyennes.

**Valeurs inhabituelles : conservées**
- 715 séjours de **0 nuit** : possibles locations à la journée (« day use »).
- 223 réservations **avec enfants mais sans adulte** : possible chambre séparée pour les enfants d'une famille.
- Valeurs extrêmes réalistes (jusqu'à 737 jours d'avance, 55 adultes pour un groupe, 69 nuits) : rares mais plausibles. Les supprimer aurait fait disparaître de vrais comportements, par exemple les groupes, qui annulent beaucoup.

**Doublons : conservés (31 994 lignes)**
C'est le choix le plus important. Le jeu de données ne contient **ni numéro de réservation ni identifiant client**. Deux lignes identiques peuvent donc être **deux vraies réservations différentes**, par exemple un groupe ou une agence qui réserve plusieurs chambres identiques le même jour. Les supprimer aurait fait passer le taux d'annulation de **37 % à environ 27 %** et aurait sous-estimé le problème, surtout pour les groupes.

**Harmonisation et formats**
- Repas « Undefined » regroupé avec « SC » (sans repas), qui a la même signification selon la documentation.
- Dates converties au format standard AAAA-MM-JJ, compatible avec BigQuery.
- Mois rangés dans l'ordre du calendrier ; `children`, `agent` et `company` passés en nombres entiers.

**Colonnes ajoutées**
- `arrival_date` : date d'arrivée complète, utile pour suivre l'évolution dans le temps.
- `arrival_month_num` : numéro du mois, pour trier correctement.
- `total_nights` et `total_guests` : durée du séjour et nombre de personnes, utiles pour l'analyse et le modèle.

**Résultat :** **119 208 réservations, 0 valeur manquante**, prêtes pour le SQL et le Machine Learning.

### 4. Ce qui était le plus important
- **Ne pas supprimer trop vite** : 99,85 % des données ont été conservées. Chaque suppression ou remplacement est justifié par le sens métier.
- **Repérer la fuite de données** : le tableau croisé montre que `reservation_status` correspond exactement à la cible (« Canceled » et « No-Show » = annulé, « Check-Out » = venu). Cette variable, ainsi que `reservation_status_date`, n'est connue qu'**après** l'arrivée prévue. Elle est gardée pour l'analyse, mais **sera exclue du modèle**, sinon il « tricherait ».
- **Comparer des taux et pas des volumes** : le City Hotel a 2 fois plus de réservations, donc forcément plus d'annulations. C'est le **taux** qui mesure le vrai risque.

### 5. Les principaux constats
- **Le City Hotel annule beaucoup plus** que le Resort Hotel (**41,8 %** contre **27,8 %**) et concentre 3 annulations sur 4.
- **Les groupes sont le segment le plus risqué** (61,1 % d'annulations), surtout au City Hotel (68,9 %).
- **Les agences hors ligne** posent un problème propre au City Hotel (42,9 % contre 15,2 % au Resort).
- **Les clients directs et les entreprises sont les plus fiables** (moins de 20 % d'annulations).

### 6. Pour la suite du projet
- Le fichier `hotel_bookings_clean.csv` est disponible dans le repo (`dataset/processed/`) pour l'**analyse SQL dans BigQuery** et le **modèle de prédiction**.
- Le modèle n'utilisera que les informations **connues au moment de la réservation**.
- Les premiers constats orientent déjà les recommandations : **surveiller en priorité les réservations de groupe et celles des agences au City Hotel**.